Objective

This notebook demonstrates how raw ArduPilot DataFlash logs are transformed into a deterministic Root Cause Analysis (RCA) pipeline.

Goal:

Detect GPS failures
Explain what happened, when, and why
Provide evidence-backed RCA with confidence

In [1]:
import os
import duckdb
import pandas as pd

DB_PATH = os.path.abspath("../bin/vault/warehouse_df/NAV_20260319_1332_parts/nav_master.duckdb")
con = duckdb.connect(DB_PATH)

print("✅ Connected to DuckDB")
print(con.execute("SHOW TABLES").fetchall())

✅ Connected to DuckDB
[('com_master',), ('context_master',), ('est_master',), ('fmt_master',), ('label_master',), ('mission_stats',), ('msg_type_master',), ('nav_ai_assistance',), ('nav_context',), ('nav_flight_context',), ('nav_master',), ('nav_mot_state_timeline',), ('nav_motion',), ('nav_rca_context',), ('nav_rca_gps_mot',), ('nav_sig_state_timeline',), ('nav_state_fc',), ('nav_state_master',), ('nav_state_timeline',), ('nav_windows',), ('power_master',), ('rule_master',), ('sys_master',)]


Step 1: GPS Availability Check
Goal: Ensure GPS messages exist in the dataset.
If missing → stop RCA (blindspot).
Expectation:GPS row count > 0

In [ ]:
anchor_df = con.execute("""
SELECT msg_type, COUNT(*)
FROM nav_master
WHERE msg_type = 'GPS'
GROUP BY msg_type;""").df()

print("✅ Anchor loaded")
display(anchor_df)

✅ Anchor loaded


,msg_type,count_star()
0,GPS,2205


Step 2: GPS Parameter Integrity Check

In [ ]:
integrity_df = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(NSats) AS nsats_present,
    COUNT(Status) AS status_present
FROM nav_master
WHERE msg_type = 'GPS';
""").df()

print("✅ GPS Parameter Integrity Check")
display(integrity_df)

✅ GPS Parameter Integrity Check


,total_rows,nsats_present,status_present
0,2205,2205,2205


Step 3: GPS Fix Type Distribution

Goal:
Understand what types of GPS fixes are present.

Reference:
0 = No GPS
1 = No Fix
2 = 2D Fix
3 = 3D Fix
4 = DGPS
5 = RTK Float
6 = RTK Fixed

Expectation:
Values should align with test scenario

In [4]:
status_dist_df = con.execute("""
SELECT Status, COUNT(*) AS count
FROM nav_master
WHERE msg_type = 'GPS'
GROUP BY Status
ORDER BY Status;
""").df()

print("✅ GPS Status Distribution")
display(status_dist_df)

✅ GPS Status Distribution


,Status,count
0,1.0,1
1,6.0,2204


Step 4: Satellite Count Behavior

Goal:
Understand how NSats varies across mission.

Expectation:
Should reflect test phases:
- High (baseline)
- Medium (degraded)
- Zero (loss)

In [5]:
nsats_dist_df = con.execute("""
SELECT NSats, COUNT(*) AS count
FROM nav_master
WHERE msg_type = 'GPS'
GROUP BY NSats
ORDER BY NSats;
""").df()

print("✅ NSats Distribution")
display(nsats_dist_df)

✅ NSats Distribution


,NSats,count
0,0.0,1301
1,3.0,1
2,7.0,56
3,12.0,847


Step 5: Sensor vs Flight Controller Consistency

Goal:
Check if NSats (sensor) and Status (FC decision) align.

Logic:
- NSats = 0 → should NOT have high Status (like 6)
- If mismatch → indicates FC lag / design issue

Output:
- ALIGNED → correct behavior
- MISMATCH → inconsistency

In [ ]:
mismatch_df = con.execute("""
SELECT
    TimeUS,
    CAST(NSats AS INTEGER) AS NSats,
    CAST(Status AS INTEGER) AS Status,

    CASE
        WHEN NSats = 0 THEN 'LOSS'
        WHEN NSats < 8 THEN 'DEGRADED'
        ELSE 'HEALTHY'
    END AS sensor_state,

    CASE
        WHEN Status <= 1 THEN 'LOSS'
        WHEN Status >= 3 THEN 'HEALTHY'
        ELSE 'DEGRADED'
    END AS fc_state,

    CASE
        WHEN
            (NSats = 0 AND Status >= 3) OR
            (NSats >= 8 AND Status <= 1)
        THEN 'MISMATCH'
        ELSE 'ALIGNED'
    END AS integrity_flag

FROM nav_master
WHERE msg_type = 'GPS'
ORDER BY TimeUS
LIMIT 100;
""").df()

print("✅ Sensor vs FC Consistency Check")
display(mismatch_df)

✅ Sensor vs FC Consistency Check


,TimeUS,NSats,Status,sensor_state,fc_state,integrity_flag
0,9819404,3,1,DEGRADED,LOSS,ALIGNED
1,9999332,0,6,LOSS,HEALTHY,MISMATCH
2,10199252,0,6,LOSS,HEALTHY,MISMATCH
3,10399172,0,6,LOSS,HEALTHY,MISMATCH
4,10599092,0,6,LOSS,HEALTHY,MISMATCH
...,...,...,...,...,...,...
95,28799309,0,6,LOSS,HEALTHY,MISMATCH
96,28999229,0,6,LOSS,HEALTHY,MISMATCH
97,29199149,0,6,LOSS,HEALTHY,MISMATCH
98,29399069,0,6,LOSS,HEALTHY,MISMATCH


Step 5: Sensor vs Flight Controller Consistency

Goal:
Check if NSats (sensor) and Status (FC decision) align.

Logic:
- NSats = 0 → should NOT have high Status (like 6)
- If mismatch → indicates FC lag / design issue

Output:
GPS Loss Windows (Sensor Truth vs FC State)
windows where:
- NSats = 0 (true GPS loss)
- Observe what FC (Status) was doing during that period
- start_time → when loss started
- end_time → when loss ended
- duration
- FC behavior during loss (min/max Status)

In [10]:
timeline_df = con.execute("""
WITH base AS (
    SELECT
        TimeUS,
        CAST(NSats AS INTEGER) AS nsats,
        CAST(Status AS INTEGER) AS status
    FROM nav_master
    WHERE msg_type = 'GPS'
),

grouped AS (
    SELECT *,
        ROW_NUMBER() OVER (ORDER BY TimeUS) -
        ROW_NUMBER() OVER (
            PARTITION BY nsats, status
            ORDER BY TimeUS
        ) AS grp
    FROM base
)

SELECT
    MIN(TimeUS) AS start_time,
    MAX(TimeUS) AS end_time,
    ROUND((MAX(TimeUS) - MIN(TimeUS)) / 1000000.0, 2) AS duration_sec,
    nsats AS nsats_value,
    status AS status_value

FROM grouped
GROUP BY grp, nsats, status
HAVING duration_sec > 0.1
ORDER BY start_time;
""").df()

print("✅ Mission Timeline: NSats vs Status")
display(timeline_df)

✅ Mission Timeline: NSats vs Status


,start_time,end_time,duration_sec,nsats_value,status_value
0,9999332,258999692,249.0,0,6
1,259199612,405799282,146.6,12,6
2,405999202,416999800,11.0,7,6
3,417199720,427999565,10.8,0,6
4,428199485,450599688,22.4,12,6


Mission Log Analysis: Identified Asynchronicity Gap

Summary:
Analysis of the mission telemetry reveals a probable gap of **asynchronicity** between the raw sensor view (`NSats`) and the Flight Controller’s (FC) reported state (`Status`). This divergence indicates a window where the "Ground Truth" and the "Internal Logic" are disconnected.

1. Dissonance Windows (Extracted from `nav_master`)

| Window | Duration (sec) | NSats (Sensor) | Status (FC) | Behavior |
|:---|:---|:---|:---|:---|
| **0** | **249.0** | **0** | **6** | Sustained asynchronicity; FC state disconnected from sensor reality. |
| **3** | **10.8** | **0** | **6** | Recurrent lag; confirms a systematic failure-reporting delay. |

2. Technical Hypothesis: The Gap
The FC advertises a "Perfect Fix" (`Status 6`) while the sensor provides zero data points (`NSats 0`). This creates a "Ghost Period" where navigation relies on an estimated state that lacks physical validation from the GPS hardware.


Investigation Process to identify the root cause of this asynchronicity.

1. **Innovation Check (nav_master):** Verify from `nav_master` (XKF3) that Innovations (IVN, IVE, IVD, IPN, IPE) remain **0.0** during the 249s window (`nsats = 0`).  
2. **Status Dump Verification:** Check the status dump to verify how the system is reporting `Status` during the same window (`status = 6`).  
3. **Parameter Linkage Check:** Identify how `nsats` is (or is not) linked to FC / estimation layer (`Status`).  
4. **State Persistence Thresholds:** Identify the logic that allows `Status 6` to persist despite total satellite loss.

Summary:
This gap serves as the primary test case for **Automated Dissonance Detection** within the **Nav-Domain: GPS Loss Analytics Pipeline**. It demonstrates the need for a secondary "Alignment Check" to ensure operational awareness of actual sensor environments.

In [ ]:
# Query to check EKF Innovations during the 'Ghost Windows'
ekf_check_df = con.execute(f"""
    SELECT
        TimeUS,
        msg_type,
        IVN, -- GPS North Velocity Innovation
        IVE, -- GPS East Velocity Innovation
        IVD, -- GPS Down Velocity Innovation
        IPN, -- GPS North Position Innovation
        IPE  -- GPS East Position Innovation
    FROM nav_master
    WHERE msg_type IN ('NKF3', 'XKF3')
    AND TimeUS BETWEEN 9999332 AND 258999692 -- Focus on Window 0
    ORDER BY TimeUS
""").df()

display(ekf_check_df)

,TimeUS,msg_type,IVN,IVE,IVD,IPN,IPE
0,9999332,XKF3,0.0,0.0,0.0,0.00,0.00
1,9999332,XKF3,0.0,0.0,0.0,0.00,0.00
2,10039316,XKF3,0.0,0.0,0.0,0.00,0.00
3,10039316,XKF3,0.0,0.0,0.0,0.00,0.00
4,10079300,XKF3,0.0,0.0,0.0,0.00,0.00
...,...,...,...,...,...,...,...
12447,258919724,XKF3,0.0,0.0,0.0,-0.03,0.04
12448,258959708,XKF3,0.0,0.0,0.0,-0.03,0.04
12449,258959708,XKF3,0.0,0.0,0.0,-0.03,0.04
12450,258999692,XKF3,0.0,0.0,0.0,-0.03,0.04


This snapshot shows Innovations at **0.0** while `Status` remains at 6 during `nsats = 0`. It highlights the need to verify system reporting and the linkage between sensor data and FC/estimation state.

![GPS Status Snapshot](/home/ni/ardupilot-nav-domain-poc/images/status_GPS.PNG)

**Tests:**

* **Test A (Signal):** We lowered the satellite count. The dashboard stayed at **Fix 6**. As long as the GPS hardware is powered on, the system ignores the fact that the data is missing.  
* **Test B (Hardware):** We turned off the GPS hardware entirely. Only then did the dashboard show **"No Fix."**

---

**The Problem:**  
The system only warns you if the hardware is "unplugged." It does not check if the information coming from that hardware is actually useful.

---

**Findings: Why the Internal Health Checks Froze**

We checked the internal health numbers (called Innovations) during that 4-minute gap. These numbers usually move up and down as the drone calculates its position.

* **The Result:** The numbers were exactly **0.0**.  
* **What that means:** Because the GPS sent zero satellites, the flight computer had nothing to compare. It didn't "fail" a test because there was no data to test. It stayed stuck in its last "Healthy" state because no new data arrived to tell it otherwise.

---

**Additional Findings:**

1. **Zero Data Response:** Changing satellite counts (nsats: from 3 to 10) results in zero change to position reporting (lat/lon) or fix quality. The FC effectively ignores the satellite count for its internal health logic.  
2. **The Missing Connection:** There is no active logic gate between "Satellite Visibility" and "Position Validity." The system maintains its last known state (Fix 6) indefinitely as long as the GPS module is powered.  
3. **The Proof:** Only a hardware-level change (`GPS1_TYPE: 0`) triggers a state change and not when SIM_GPS1_NUMSATS values are set.

**Issue: The Gap Between Sensor and Status**  
It shows how the drone can report it is in a "Perfect" state (status=6) even when it has no GPS signal (nsats=0) at all.